# 07 — Predict delivery times for unlabelled test rows
This notebook uses the trained model and recreates the feature transformations for the test dataset. It does not evaluate accuracy because the test file has no true target.

In [1]:
from pathlib import Path
import re, json, joblib
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
RAW, PROCESSED, MODELS, REPORTS = ROOT/"data/raw", ROOT/"data/processed", ROOT/"models", ROOT/"reports"
model = joblib.load(MODELS/"delivery_eta.joblib")
meta = json.loads((MODELS/"delivery_eta_metadata.json").read_text())
feature_columns = meta["feature_columns"]

def parse_order_time(value):
    if pd.isna(value): return np.nan, np.nan
    m = re.match(r"^\s*(\d{1,2}):(\d{2})(?::\d{2})?\s*(AM|PM)?\s*$", str(value), re.I)
    if not m: return np.nan, np.nan
    hour, minute = int(m.group(1)), int(m.group(2))
    suffix = (m.group(3) or "").upper()
    if suffix == "PM" and hour < 12: hour += 12
    if suffix == "AM" and hour == 12: hour = 0
    if hour > 23 or minute > 59: return np.nan, np.nan
    return hour, minute

def build_features(df):
    x = df.copy()
    numeric = ["Delivery_person_Age", "Delivery_person_Ratings", "Restaurant_latitude",
               "Restaurant_longitude", "Delivery_location_latitude",
               "Delivery_location_longitude", "Vehicle_condition"]
    for c in numeric:
        x[c] = pd.to_numeric(x[c], errors="coerce")
    lat1, lon1 = np.radians(x["Restaurant_latitude"]), np.radians(x["Restaurant_longitude"])
    lat2, lon2 = np.radians(x["Delivery_location_latitude"]), np.radians(x["Delivery_location_longitude"])
    a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
    x["distance_km"] = 6371 * 2*np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    hm = x["Time_Orderd"].apply(parse_order_time)
    x["order_hour"] = hm.map(lambda v: v[0])
    x["order_minute"] = hm.map(lambda v: v[1])
    dt = pd.to_datetime(x["Order_Date"], errors="coerce", dayfirst=True)
    x["order_weekday"] = dt.dt.dayofweek
    x["order_month"] = dt.dt.month
    cats = ["Weatherconditions", "Road_traffic_density", "Type_of_order",
            "Type_of_vehicle", "Festival", "City"]
    for c in cats:
        x[c] = x[c].astype("string").str.strip().replace(
            {"": pd.NA, "NaN": pd.NA, "nan": pd.NA, "conditions NaN": pd.NA}
        ).astype(object)
        x[c] = x[c].where(pd.notna(x[c]), np.nan)
    return x[feature_columns]

test_path = RAW/"delivery_time_test.csv"
test = pd.read_csv(test_path, low_memory=False)
X_test = build_features(test)
pred = np.clip(model.predict(X_test), 1, 180)
out = pd.DataFrame()
if "ID" in test:
    out["ID"] = test["ID"]
out["predicted_delivery_minutes"] = np.round(pred, 1)
out.to_csv(REPORTS/"delivery_test_predictions.csv", index=False)
display(out.head())
print("Saved:", REPORTS/"delivery_test_predictions.csv")

/var/folders/44/fywnm5sj335fdzvrq6s4_m3c0000gn/T/ipykernel_4098/3141325802.py:39: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  x["order_weekday"] = dt.dt.dayofweek


,ID,predicted_delivery_minutes
0,0x2318,27.2
1,0x3474,29.8
2,0x9420,27.7
3,0x72ee,31.3
4,0xa759,21.0


Saved: /Users/kakulrajput/Documents/food-ordering-delivery-ml/reports/delivery_test_predictions.csv
